# 07. Causal Representation Identifiability: iVAE, Synthetic Control, and Symbolic Regression

- **Category:** 09_causal_ai_and_evolutionary_search
- **Lineage & Citations:** Abadie et al. (2010 *Synthetic Control Methods*); Locatello et al. (2019 *Challenging Common Assumptions in the Unsupervised Learning of Disentangled Representations*); Khemakhem et al. (2020 *ICE-BEE / iVAE: Variational Autoencoders and Nonlinear ICA: A Theoretical Framework*); Cranmer (2023 *PySR*).
- **Purpose:** Establish mathematically identifiable causal latent factor recovery via auxiliary conditioning, estimate unit-level counterfactual policy effects via simplex-constrained synthetic controls, and discover closed-form scientific equations via Pareto multi-objective search.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. Population Based Training and DARTS](06_population_based_training_pbt_and_darts_neural_architecture_search.ipynb) $\to$ `🔗 Downstream Syntheses:` [10. XAI Local Surrogates LIME and SHAP](../10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use unconstrained linear regression for synthetic control estimation without the simplex constraint $w_j \ge 0, \sum w_j = 1$ (creates massive out-of-hull extrapolation bias); do not expect standard unsupervised VAEs to recover true causal factors without auxiliary conditioning $u$.

## Mathematical Foundations & Governing Equations

1. **iVAE Conditional Factorial Gaussian KL**: $D_{\text{KL}}(q_\phi(z|x, u) \parallel p_\theta(z|u)) = \frac{1}{2} \sum_j \left[ \frac{\sigma_{z, j}^2 + (\mu_{z, j} - \mu_{u, j})^2}{\sigma_{u, j}^2} - 1 - (\log \sigma_{z, j}^2 - \log \sigma_{u, j}^2) \right]$
2. **Simplex-Constrained Synthetic Control**: $\min_{w \in \Delta^J} \|X_1 - X_0 w\|_2^2 \quad \text{s.t.} \quad w_j \ge 0, \; \sum w_j = 1$
3. **Causal Treatment Effect**: $\hat{\tau}_t = Y_{1, t} - \sum_j w_j Y_{0, t, j}$
4. **Symbolic Regression Pareto Loss**: $\mathcal{L}_{\text{Pareto}} = \text{MSE}(y, \hat{y}) + \lambda \cdot \text{Complexity}(f)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def ivae_kl(mu_z, logvar_z, mu_u, logvar_u):
    var_z, var_u = torch.exp(logvar_z), torch.exp(logvar_u)
    kl = 0.5 * ((var_z + (mu_z - mu_u)**2) / var_u - 1.0 - (logvar_z - logvar_u))
    return torch.sum(kl, dim=-1)

def project_simplex(v):
    u, _ = torch.sort(v, descending=True)
    cssv = torch.cumsum(u, dim=0) - 1.0
    ind = torch.arange(1, v.shape[0] + 1, dtype=v.dtype, device=v.device)
    rho = torch.max(torch.where(u - cssv / ind > 0.0, ind, torch.zeros_like(ind))).long()
    theta = cssv[rho - 1] / float(rho)
    return torch.clamp(v - theta, min=0.0)

def synthetic_control(y1, y0, w):
    synth = y0 @ w
    return synth, y1 - synth

def symbolic_pareto(y_true, y_pred, complexity, lambda_reg):
    return torch.mean((y_true - y_pred)**2) + lambda_reg * complexity

In [ ]:
# Parity verification against golden reference tensors
kl = ivae_kl(weights["mu_z"], weights["logvar_z"], weights["mu_u"], weights["logvar_u"])
assert torch.max(torch.abs(kl - weights["ivae_kl_ref"])) < 1e-5, "iVAE KL mismatch"

w_simp = project_simplex(weights["sc_v"])
assert torch.max(torch.abs(w_simp - weights["sc_w_simplex_ref"])) < 1e-5, "Simplex mismatch"

synth, tau = synthetic_control(weights["sc_y1_post"], weights["sc_y0_post"], w_simp)
assert torch.max(torch.abs(synth - weights["sc_y1_synth_ref"])) < 1e-5, "Synthetic control mismatch"
assert torch.max(torch.abs(tau - weights["sc_tau_ref"])) < 1e-5, "Treatment effect mismatch"

pareto = symbolic_pareto(weights["sr_y_true"], weights["sr_y_pred"], weights["sr_complexity"], weights["sr_lambda"])
assert torch.max(torch.abs(pareto - weights["sr_pareto_loss_ref"])) < 1e-5, "Pareto loss mismatch"
print("All iVAE, Synthetic Control, and Symbolic Regression parity checks passed!")

In [ ]:
# Benchmarking causal identifiability and synthetic control operators
t0 = time.perf_counter()
for _ in range(500):
    _ = ivae_kl(weights["mu_z"], weights["logvar_z"], weights["mu_u"], weights["logvar_u"])
    w_s = project_simplex(weights["sc_v"])
    _ = synthetic_control(weights["sc_y1_post"], weights["sc_y0_post"], w_s)
    _ = symbolic_pareto(weights["sr_y_true"], weights["sr_y_pred"], weights["sr_complexity"], weights["sr_lambda"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused iVAE + Simplex + Synthetic Control latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Unconstrained OLS vs Simplex-Constrained Synthetic Control
# When fitting pre-intervention treated trajectory X1 from donor pool X0:
X1 = torch.tensor([10.0, 12.0, 15.0])
X0 = torch.tensor([[8.0, 14.0], [10.0, 16.0], [12.0, 18.0]]) # Donors
# Unconstrained OLS: min ||X1 - X0 w||^2 -> allows negative weights and extrapolation
w_ols = torch.linalg.lstsq(X0, X1).solution
# Simplex projected weights
w_simplex = project_simplex(w_ols)
print(f"Unconstrained OLS weights: {w_ols.tolist()} (contains negative weight {w_ols[1]:.2f})")
print(f"Simplex-constrained weights: {w_simplex.tolist()} (strictly non-negative, sum={w_simplex.sum():.2f})")
assert (w_ols < 0.0).any(), "OLS must exhibit unconstrained negative weights"
assert (w_simplex >= 0.0).all() and abs(w_simplex.sum().item() - 1.0) < 1e-5, "Simplex weights must be valid probability distribution"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **iVAE Identifiability**: Auxiliary conditioning $u$ breaks rotational invariance in the latent prior, mathematically guaranteeing causal factor disentanglement.
2. **Simplex Synthetic Control**: Constraining donor weights to the probability simplex guarantees convex interpolation and eliminates out-of-hull extrapolation bias.
3. **Symbolic Pareto Frontiers**: Balancing goodness-of-fit against expression complexity prevents algebraic bloat.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.md](../../markdown/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.md)
- **PyTorch 2.x Module:** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.py](../../pytorch2/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.py)
- **Functional JAX Module:** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.py](../../jax/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.py)
- **Native Rust Candle Track:** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.rs](../../candle/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.rs)
- **Hardware Kernel (CPU / WGSL):** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.rs](../../kernels/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.rs)
- **Golden SafeTensors:** [07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.safetensors](../../weights/09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Population Based Training PBT and DARTS Neural Architecture Search](06_population_based_training_pbt_and_darts_neural_architecture_search.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | Next: &rarr;